# Maestría en Maestría en Ciencia de Datos e Inteligencia Artificial
#### 8. Machine Learning and Deep Learning
#### Docente: Msc. Renzo Claure Aracena.

In [ ]:
import os
os.getcwd()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, InputLayer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import History
import pickle  # Para guardar las métricas
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score, precision_score
#--------
from scikeras.wrappers import KerasClassifier #Para que funcione se debe instalar pip install scikeras
#-------

In [ ]:
#Importar los datos
data = load_breast_cancer()
X = data.data
y = data.target

In [ ]:
#Muestreamos para ent. y comp.
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

In [ ]:
#Normalizamos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Crear el modelo, es más conveniente en una función
def create_model(neurons=16, dropout_rate=0.2, learning_rate=0.001):
    #Creación del modelo de tensorflow
    model = Sequential()
    
    #Capa de entrada
    #dimension de entrada a esta capa = cantidad de variables ind.
    model.add(InputLayer(shape=(X_test_scaled.shape[1],)))
    
    #Crear primera capa oculta
    #capa con 16 neuronas, 
    model.add(Dense(neurons, activation='relu')) #funcióm de activación 'relu'
    model.add(Dropout(dropout_rate)) #Función de regularización 20%

    #Segunda capa oculta
    model.add(Dense(neurons*2, activation='relu'))
    
    #Capa de salida
    model.add(Dense(1, activation='sigmoid')) #se reduce a 1 neurona, y se usa sigmoide ya que nos dará una probabilidad

    #Configuraciónn del optimizer y compilación del modelo
    optimizer = Adam(learning_rate= learning_rate) #Cuidado con este parámetro
    
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

In [ ]:
#Para poder hacer Gridsearch con CV se utiliza un model Wrapper
#model_wrapper = KerasClassifier(builfn=create_model, verborse=0)

In [ ]:
#Para poder usar esta seccion se debe hacer un downgrade de sklearn a 1.3 - 1.5
#Definimos el Gris de hiperparámetros
#param_grid = {
#    'neurons' : [16, 32],
#    'dropout_rate' : [0.2, 0.3],
#    'learning_rate' : [0.001, 0.0001],
#    'batch_size': [16, 32],
#    'epochs': [50, 100]
#}

In [ ]:
# Definimos el Grid de hiperparámetros
#grid = GridSearchCV(estimator=model_wrapper, param_grid=param_grid, cv=3, scoring='accuracy')
#grid_result = grid.fit(X_train_scaled, y_train)
# Mostrar los mejores resultados y parámetros
#print("Mejor precisión: {:.4f} usando {}".format(grid_result.best_score_, grid_result.best_params_))

# Evaluar el modelo en el conjunto de prueba
#best_model_wrapper = grid_result.best_estimator_
# Realizar predicciones en el conjunto de prueba
#y_pred = best_model_wrapper.predict(X_test)

In [ ]:
#Crear y entrenar el modelo sin GridSearch
model = create_model()
history = model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled, y_test), verbose=1)

In [ ]:
#Evaluar el modelo en el conjunto de prueba
y_pred = (model.predict(X_test_scaled) > 0.5).astype("int32")
loss, accuracy = model.evaluate(X_test_scaled, y_test)

# Calcular y mostrar métricas
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))

In [ ]:
#Guardar el modelo
model.save("modelo_cancer.keras")
print("\nModelo guardado como 'modelo_cancer.keras'")

In [ ]:
metrics = {
    "loss": loss,
    "accuracy": accuracy,
    "history": history.history  # Guarda el historial completo
}

In [ ]:
with open("metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)

print("\nModelo y métricas guardadas")

In [ ]:
#Ejemplo de uso en producción
#Supongamos que se tienen nuevos datos
# 2. Cargar el dataset para simular nuevos datos
data = load_breast_cancer()
X = data.data
y = data.target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
_, X_test, _, _ = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
#sample = X_test[0].reshape(1, -1)
sample = np.array(X_test[0].reshape(1, -1), dtype=np.float32)

In [ ]:
#Cargar el modelo
model_load = load_model("modelo_cancer.keras")
print("\nModelo cargado!")

#Cargar las métricas desde el archivo
with open("metrics.pkl", "rb") as f:
    loaded_metrics = pickle.load(f)

In [ ]:
print("\nMétricas guardadas:")
print("Loss:", loaded_metrics["loss"])
print("Accuracy:", loaded_metrics["accuracy"])

In [ ]:
#Acceder al historial de entrenamiento
print("\nHistorial de entrenamiento (últimos 5 epochs):")
for key, values in loaded_metrics["history"].items():
    print(f"{key}: {values[-5:]}")  # Últimos 5 valores

In [ ]:
#Usar el modelo
pred_sample = model_load.predict(sample)
predicted_class = (pred_sample > 0.5).astype('int32')

In [ ]:
print("\nPredicción de la muestra:", predicted_class[0][0])  